# Import Library and Dataset

In [1]:
# Import essential library
import pandas as pd
import numpy as np
import re

In [2]:
# Import datasets
general_df = pd.read_csv("goemotions_mrm8488_train.csv")
domain_df = pd.read_csv("stockemotions_full.csv", skiprows = 1)

print(general_df.shape)
print(domain_df.shape)
print(general_df.head(10))
print(domain_df.head(10))

(211225, 37)
(10000, 7)
                                                text       id  \
0                                    That game hurt.  eew5j0j   
1   >sexuality shouldn’t be a grouping category I...  eemcysk   
2     You do right, if you don't care then fuck 'em!  ed2mah1   
3                                 Man I love reddit.  eeibobj   
4  [NAME] was nowhere near them, he was by the Fa...  eda6yn6   
5  Right? Considering it’s such an important docu...  eespn2i   
6  He isn't as big, but he's still quite popular....  eczuekb   
7  That's crazy; I went to a super [RELIGION] hig...  ed5tx8y   
8                                that's adorable asf  ef961hv   
9  "Sponge Blurb Pubs Quaw Haha GURR ha AAa!" fin...  edl7cr3   

                author             subreddit    link_id   parent_id  \
0                Brdd9                   nrl  t3_ajis4z  t1_eew18eq   
1          TheGreen888      unpopularopinion  t3_ai4q37   t3_ai4q37   
2             Labalool           confessions  t

# Basic Statistics


In [3]:
# The shape of the datasets
print(f'General Dataset: {general_df.shape}')
print(f'Domain Dataset: {domain_df.shape}')

General Dataset: (211225, 37)
Domain Dataset: (10000, 7)


In [4]:
# Check datatype
print(general_df.info())
print("________________________________\n")
print(domain_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211225 entries, 0 to 211224
Data columns (total 37 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   text                  211225 non-null  object 
 1   id                    211225 non-null  object 
 2   author                211225 non-null  object 
 3   subreddit             211225 non-null  object 
 4   link_id               211225 non-null  object 
 5   parent_id             211225 non-null  object 
 6   created_utc           211225 non-null  float64
 7   rater_id              211225 non-null  int64  
 8   example_very_unclear  211225 non-null  bool   
 9   admiration            211225 non-null  int64  
 10  amusement             211225 non-null  int64  
 11  anger                 211225 non-null  int64  
 12  annoyance             211225 non-null  int64  
 13  approval              211225 non-null  int64  
 14  caring                211225 non-null  int64  
 15  

In [5]:
# Checking missing value
print(f'General Dataset: {general_df.isnull().sum()}')
print("________________________________\n")
print(f'Domain Dataset: {domain_df.isnull().sum()}')

General Dataset: text                    0
id                      0
author                  0
subreddit               0
link_id                 0
parent_id               0
created_utc             0
rater_id                0
example_very_unclear    0
admiration              0
amusement               0
anger                   0
annoyance               0
approval                0
caring                  0
confusion               0
curiosity               0
desire                  0
disappointment          0
disapproval             0
disgust                 0
embarrassment           0
excitement              0
fear                    0
gratitude               0
grief                   0
joy                     0
love                    0
nervousness             0
optimism                0
pride                   0
realization             0
relief                  0
remorse                 0
sadness                 0
surprise                0
neutral                 0
dtype: int64
________

# Clean Dataset


## General Dataset - GoEmotion
viết cho tôi 1 đoạn văn ngắn để tôi nhét vào vscode giải thích bước tôi làm clean dataset go emtion. đầu tiên filter cột "example_very_unclear" = True thì bỏ, sau đó row nào có nhiều nhãn thì

In [6]:

# Fiter ()"example_very_unclear" = True) as it do not have any label
general_df = general_df[~general_df["example_very_unclear"]]
print(general_df["example_very_unclear"].head(10))

0     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
Name: example_very_unclear, dtype: bool


In [7]:
# Applying rarest-label strategy to balance the dataset
min_vote = 2
emotion_cols = general_df.columns[9:].tolist()
agg = general_df.groupby('text')[emotion_cols].sum()
mask = agg >= min_vote
agg = agg[mask.any(axis = 1)]
mask = mask[mask.any(axis = 1)]
rare_rank = mask.sum().rank(method = "first")
scores = (mask * rare_rank).where(mask)
agg["label"] = scores.idxmin(axis=1)
general_clean = agg.reset_index()[["text", "label"]]
print(general_clean.head(10))
print(general_clean.shape)


                                                text        label
0   "If you don't wear BROWN AND ORANGE...YOU DON...    annoyance
1   "What do Scottish people look like?" How I wo...         love
2     ### A surprise, to be sure, but a welcome one      surprise
3   '*Pray*, v. To ask that the laws of the unive...      neutral
4   >it'll get invaded by tankie, unfortunately. ...      neutral
5   And not all children's hospitals need the sam...     approval
6                Best number! [NAME], [NAME], [NAME]   admiration
7   Calm down and relax are the worst things to s...    annoyance
8   Change is hard. Find comfort in victory, even...      neutral
9   Don't be so stupid. Terrorism is inherently p...  disapproval
(53994, 2)


In [8]:
duplicate_value = general_clean[general_clean.duplicated("text", keep=False)].sort_values("text")
print("Duplicate Rows:", len(duplicate_value), "| Duplicate Text", duplicate_value["text"].nunique())

Duplicate Rows: 0 | Duplicate Text 0


## Domain Dataset - StockEmotions

In [9]:
#Check duplicate value
duplicate_data = domain_df[domain_df.duplicated()]
print(duplicate_value)

Empty DataFrame
Columns: [text, label]
Index: []


In [10]:
domain_df["text"] = (domain_df["processed"].astype(str).str.replace(r"[\u200d\ufe0f]", "", regex=True).str.replace(r"[\U0001F000-\U0001FAFF\U0001F1E6-\U0001F1FF\u2190-\u21FF\u2600-\u27BF\u2B00-\u2BFF]", " ", regex=True).str.replace(r"\$(?=[A-Za-z])", "", regex=True).str.replace(r"\s+", " ", regex=True).str.strip())
domain_df.to_csv("domain_df.csv", index = True, encoding = "utf-8")
domain_clean = domain_df[["text", "emo_label"]].rename(columns={"emo_label": "label"})
print(domain_clean.head(10))

                                                text       label
0  Amazon Dow futures up by 100 points already [p...  excitement
1  Tesla Daddy's drinkin' eArly tonight! Here's t...  excitement
2  Apple We’ll been riding since last December fr...   confusion
3  Tesla happy new year, 2020, everyone [wine gla...  excitement
4  Tesla haha just a collection of greats..."Mars...  excitement
5  Tesla NOBODY: Gas cars driven by humans killed...    surprise
6  Apple $300 calls First trade of 2020 Congrats ...   amusement
7  Apple Remember, if you short every day, one of...     anxiety
8  Apple called it, the bear comment below makes ...    optimism
9  Home Depot Bought more at today's low. She is ...    optimism

Phase 3

## Build Sentiment Labels
GoEmotions' 28 emotion labels are mapped to binary sentiment (positive/negative) using the standard valence grouping. Ambiguous/neutral emotions (neutral, confusion, curiosity, realization, surprise) are dropped since the domain dataset has no matching neutral class. StockEmotions' `senti_label` (bullish/bearish) maps directly to the same scale.

In [11]:
# Map GoEmotions' 28 emotions to binary sentiment; ambiguous/neutral emotions are dropped
positive_emotions = {"admiration", "amusement", "approval", "caring", "desire", "excitement",
                      "gratitude", "joy", "love", "optimism", "pride", "relief"}
negative_emotions = {"anger", "annoyance", "disappointment", "disapproval", "disgust",
                      "embarrassment", "fear", "grief", "nervousness", "remorse", "sadness"}

def map_emotion_to_sentiment(label):
    if label in positive_emotions:
        return "positive"
    if label in negative_emotions:
        return "negative"
    return None

general_clean["sentiment"] = general_clean["label"].map(map_emotion_to_sentiment)
general_clean = general_clean.dropna(subset=["sentiment"]).reset_index(drop=True)

print(general_clean["sentiment"].value_counts())
print(general_clean.shape)

sentiment
positive    20134
negative    11733
Name: count, dtype: int64


(31867, 3)


In [12]:
# Map domain senti_label (bullish/bearish) to the same sentiment scale
sentiment_map = {"bullish": "positive", "bearish": "negative"}
domain_df["sentiment"] = domain_df["senti_label"].map(sentiment_map)

print(domain_df["sentiment"].value_counts())

sentiment
positive    5474
negative    4526
Name: count, dtype: int64


## Train/Test Split for Domain Dataset
A single domain test set is held out and reused across Models A, B, and C so the three results are directly comparable.

In [13]:
from sklearn.model_selection import train_test_split

domain_train, domain_test = train_test_split(
    domain_df[["text", "sentiment"]],
    test_size=0.2,
    stratify=domain_df["sentiment"],
    random_state=42
)

print(f"Domain train: {domain_train.shape}, Domain test: {domain_test.shape}")

Domain train: (8000, 2), Domain test: (2000, 2)


## TF-IDF + Logistic Regression Helper
The same model type (Logistic Regression) and TF-IDF setup are reused for Models A, B, and C — only the training data changes. Each model fits its own TF-IDF vectorizer on its own training text.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

def train_and_evaluate(train_texts, train_labels, test_texts, test_labels, model_name):
    vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, train_labels)
    preds = clf.predict(X_test)

    acc = accuracy_score(test_labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(test_labels, preds, average="weighted")
    cm = confusion_matrix(test_labels, preds, labels=["positive", "negative"])

    print(f"=== {model_name} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print("Confusion Matrix [rows=true, cols=pred] (positive, negative):")
    print(cm)
    print(classification_report(test_labels, preds))

    return {"model": model_name, "accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

## Model A — General Only (No Domain Adaptation)
Trained on the general dataset, tested on the held-out domain test set.

In [15]:
results_A = train_and_evaluate(
    train_texts=general_clean["text"],
    train_labels=general_clean["sentiment"],
    test_texts=domain_test["text"],
    test_labels=domain_test["sentiment"],
    model_name="Model A - General Only"
)

=== Model A - General Only ===
Accuracy:  0.5410
Precision: 0.5241
Recall:    0.5410
F1-score:  0.5007
Confusion Matrix [rows=true, cols=pred] (positive, negative):
[[872 223]
 [695 210]]
              precision    recall  f1-score   support

    negative       0.48      0.23      0.31       905
    positive       0.56      0.80      0.66      1095

    accuracy                           0.54      2000
   macro avg       0.52      0.51      0.48      2000
weighted avg       0.52      0.54      0.50      2000



## Model B — Domain Only (No Transfer Learning)
Trained on the domain train split only, tested on the domain test split.

In [16]:
results_B = train_and_evaluate(
    train_texts=domain_train["text"],
    train_labels=domain_train["sentiment"],
    test_texts=domain_test["text"],
    test_labels=domain_test["sentiment"],
    model_name="Model B - Domain Only"
)

=== Model B - Domain Only ===
Accuracy:  0.7575
Precision: 0.7570
Recall:    0.7575
F1-score:  0.7570
Confusion Matrix [rows=true, cols=pred] (positive, negative):
[[871 224]
 [261 644]]
              precision    recall  f1-score   support

    negative       0.74      0.71      0.73       905
    positive       0.77      0.80      0.78      1095

    accuracy                           0.76      2000
   macro avg       0.76      0.75      0.75      2000
weighted avg       0.76      0.76      0.76      2000



## Model C — Mixed Training (General + Domain)
Trained on the general dataset combined with the domain train split, tested on the same domain test split as Models A and B.

In [17]:
mixed_train_texts = pd.concat([general_clean["text"], domain_train["text"]], ignore_index=True)
mixed_train_labels = pd.concat([general_clean["sentiment"], domain_train["sentiment"]], ignore_index=True)

results_C = train_and_evaluate(
    train_texts=mixed_train_texts,
    train_labels=mixed_train_labels,
    test_texts=domain_test["text"],
    test_labels=domain_test["sentiment"],
    model_name="Model C - Mixed Training"
)

=== Model C - Mixed Training ===
Accuracy:  0.7335
Precision: 0.7332
Recall:    0.7335
F1-score:  0.7333
Confusion Matrix [rows=true, cols=pred] (positive, negative):
[[835 260]
 [273 632]]
              precision    recall  f1-score   support

    negative       0.71      0.70      0.70       905
    positive       0.75      0.76      0.76      1095

    accuracy                           0.73      2000
   macro avg       0.73      0.73      0.73      2000
weighted avg       0.73      0.73      0.73      2000



## Model Comparison Summary

In [18]:
summary_df = pd.DataFrame([results_A, results_B, results_C])
print(summary_df)

                      model  accuracy  precision  recall        f1
0    Model A - General Only    0.5410   0.524129  0.5410  0.500733
1     Model B - Domain Only    0.7575   0.756991  0.7575  0.756984
2  Model C - Mixed Training    0.7335   0.733207  0.7335  0.733322
